In [ ]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt

# import evaluate_samples function from /work/work_fran/sampling/experiments/utils/evaluate.py
# import sys
# sys.path.append("/work/work_fran/sampling/experiments")
# from utils.evaluate import evaluate_samples

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print("Using device:", DEVICE)

In [ ]:
from sampling.targets import GaussianMixture

def imshow_density(log_prob, bins: int, scale: float, ax = None, **kwargs):
    if ax is None:
        ax = plt.gca()
    x = torch.linspace(-scale, scale, bins).to(DEVICE)
    y = torch.linspace(-scale, scale, bins).to(DEVICE)
    X, Y = torch.meshgrid(x, y)
    xy = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=-1)
    density = log_prob(xy).reshape(bins, bins).T
    # print("Density:", density.min().item(), density.max().item())
    im = ax.imshow(density.cpu(), extent=[-scale, scale, -scale, scale], origin='lower', **kwargs)

def contour_density(log_prob, bins: int, scale: float, ax = None, **kwargs):
    if ax is None:
        ax = plt.gca()
    x = torch.linspace(-scale, scale, bins).to(DEVICE)
    y = torch.linspace(-scale, scale, bins).to(DEVICE)
    X, Y = torch.meshgrid(x, y)
    xy = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=-1)
    density = log_prob(xy).reshape(bins, bins).T
    im = ax.contour(density.cpu(), extent=[-scale, scale, -scale, scale], origin='lower', **kwargs)

def uniform_gaussian_mixture(
    scale: float = 40.0, 
    dim: int = 2,
    log_var_scaling: float = 1.0,
    n_mixes: int = 40,
    seed: int = 0
) -> GaussianMixture:
    generator = torch.Generator(device="cpu").manual_seed(seed)
    generator = torch.Generator(device="cpu").manual_seed(seed)
    mean = (torch.rand((n_mixes, dim), generator=generator) - 0.5) * 2 * scale
    log_var = torch.ones((n_mixes, dim)) * log_var_scaling
    covs = torch.diag_embed(torch.nn.functional.softplus(log_var))
    weights = torch.ones((n_mixes,))
    return GaussianMixture(mean, covs, weights)

def nonuniform_gaussian_mixture(
    scale: float = 40.0, 
    dim: int = 2,
    log_var_scaling: float = 1.0,
    n_mixes: int = 40,
    seed: int = 0,
    perc_big_mixes: float = 0.15,
    weight_big_mixes: float = 20.0
) -> GaussianMixture:
    generator = torch.Generator(device="cpu").manual_seed(seed)
    generator = torch.Generator(device="cpu").manual_seed(seed)
    mean = (torch.rand((n_mixes, dim), generator=generator) - 0.5) * 2 * scale
    log_var = torch.ones((n_mixes, dim)) * log_var_scaling
    covs = torch.diag_embed(torch.nn.functional.softplus(log_var))
    weights = torch.ones((n_mixes,))
    n_big_mixes = int(n_mixes * perc_big_mixes)
    weights[:n_big_mixes] *= weight_big_mixes
    return GaussianMixture(mean, covs, weights)

In [ ]:
from sampling.targets import ManyWell

scale = 40.0
TARGET = nonuniform_gaussian_mixture(scale=scale).to(DEVICE)
# TARGET = uniform_gaussian_mixture(scale=scale).to(device)
# TARGET = ManyWell(dim=8).to(device)
EPS = 1e-6
N_SAMPLES = 10000
gt_samples = TARGET.sample(N_SAMPLES).to(DEVICE)

SAVE_PATH = "/home/fran/work_fran/sampling/experiments/figures/"

In [ ]:
def plot_results(ax, samples):
    bins = 200
    scale_plot = scale * 1.3

    log_prob = lambda x : TARGET.log_prob(x)
    gt_samples = torch.clamp(samples, -scale*1.5, scale*1.5)

    
    imshow_density(log_prob, bins, scale_plot, ax, cmap=plt.get_cmap('Blues'), vmin=-scale, alpha=0.7)
    contour_density(log_prob, bins, scale_plot, ax, colors='grey', linestyles='solid', alpha=0.25, levels=20)
    ax.scatter(
        samples[:, 0].cpu(), 
        samples[:, 1].cpu(), 
        s=1, alpha=0.5, color='black', rasterized=True
    )
    ax.set_xticks([])
    ax.set_yticks([])


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
plot_results(ax, gt_samples)
plt.tight_layout()
plt.savefig(SAVE_PATH + "samples_gmnu2_gt.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def build_schedule(min_val, max_val, n_steps):
    r = (max_val / min_val) ** (1.0 / (n_steps - 1))
    schedule = min_val * (r ** torch.arange(n_steps))
    return schedule

In [ ]:
from sampling.samplers import PT, ProgressiveInterpolationSDE, MALA, HMC, DiGS, SMC
from sampling.kernels import MALAKernel


COMPILE = True
N_STEPS = 1000

x0 = torch.zeros((N_SAMPLES, TARGET.dim)).to(DEVICE)

def build_sampler(name):
    if name == "parallel_tempering":
        
        n_replicas = 5
        beta_schedule = build_schedule(0.01, 1.0, n_replicas).to(DEVICE)

        kernel = MALAKernel()

        sampler = PT(
            target=TARGET,
            kernel=kernel,
            beta_schedule=beta_schedule,
            step_size=0.1,
            verbose=True,
            compile=COMPILE
        ).to(DEVICE)
    elif name == "progressive_interpolation_sde":
        
        n_replicas = 5
        int_steps = 10
        jump_steps = math.ceil(N_STEPS  - int_steps) // n_replicas

        time_schedule = build_schedule(0.01, 1.0, int_steps+2).to(DEVICE)
        base_noise_var = 0.5
        noise_schedule = torch.full_like(time_schedule, base_noise_var)
        jump_beta_schedule = build_schedule(0.01, 1.0, n_replicas).to(DEVICE)

        sampler = ProgressiveInterpolationSDE(
            target=TARGET,
            time_schedule=time_schedule,
            noise_schedule=noise_schedule,
            corrector_mode="mala",
            corrector_steps=1,
            corrector_adaptation_rate=0.0,
            corrector_step_size=0.1,
            jump_ref_std=1.0,
            jump_steps=jump_steps,
            jump_beta_schedule=jump_beta_schedule,
            compile=COMPILE,
            verbose=True
        ).to(DEVICE)
    elif name == "smc":
        
        n_particles = 4
        n_steps_smc = N_STEPS // n_particles
        beta_schedule = build_schedule(0.01, 1.0, n_steps_smc).to(DEVICE)
        kernel = MALAKernel()

        sampler = SMC(
            target=TARGET,
            kernel=kernel,
            beta_schedule=beta_schedule,
            n_particles=n_particles,
            step_size=0.1,
            resampling_threshold=1.0,
            verbose=True,
            compile=COMPILE
        ).to(DEVICE)
    
    elif name == "digs":

        n_noise_levels = 1
        n_denoising_steps = 1
        n_steps_digs = N_STEPS // (1 + n_denoising_steps)
        n_gibbs_sweeps = n_steps_digs // n_noise_levels

        alpha_shedule = torch.linspace(0.1, 0.9, n_noise_levels)
        sigma_schedule = torch.sqrt(1.0 - alpha_shedule ** 2)

        sampler = DiGS(
            target=TARGET,
            denoising_kernel=MALAKernel(),
            alpha_schedule=alpha_shedule.to(DEVICE),
            sigma_schedule=sigma_schedule.to(DEVICE),
            n_denoising_steps=n_denoising_steps,
            n_gibbs_sweeps=n_gibbs_sweeps,
            denoising_step_size=0.1,
            compile=COMPILE,
            verbose=True,
        )
    
    elif name == "mala":
        sampler = MALA(
            target=TARGET,
            step_size=0.1,
            verbose=True,
            compile=COMPILE,
        ).to(DEVICE)
    elif name == "hmc":
        sampler = HMC(
            target=TARGET,
            step_size=0.1,
            n_leapfrog_steps=3,
            verbose=True,
            compile=COMPILE,
        ).to(DEVICE)
    else:
        raise ValueError(f"Unknown sampler: {name}")
    
    return sampler
    

def run_sampler(sampler, name):
    if name == "parallel_tempering":
        n_steps = N_STEPS // sampler.beta_schedule.shape[0]
        samples = sampler(
            x0=x0,
            n_steps=n_steps
        )
        samples = samples[:, -1, :]
    elif name == "progressive_interpolation_sde":
        z = x0.clone()
        samples = sampler(
            x0=x0,
            z=z
        )
    elif name == "smc":
        samples = sampler(
            x0=x0
        )
    elif name == "digs":
        samples = sampler(
            x0=x0,
        )
    elif name in ["mala", "hmc"]:
        n_steps = N_STEPS
        samples = sampler(
            x0=x0,
            n_steps=n_steps
        )
    else:
        raise ValueError(f"Unknown sampler: {name}")

    return samples


In [ ]:

samplers_to_run = [
    "progressive_interpolation_sde",
    "parallel_tempering",
    "smc",
    "digs",
    "hmc",
    "mala"
]

samples_dict = {}

for sampler_name in samplers_to_run:
    print(f"Running sampler: {sampler_name}")

    sampler = build_sampler(sampler_name)
    samples = run_sampler(sampler, sampler_name)
    samples_dict[sampler_name] = samples.clone()

In [ ]:
NAMES_TO_SHOW = {
    "gt": "Ground Truth",
    "parallel_tempering" : "NRPT", 
    "progressive_interpolation_sde" : "CDS (ours)",
    "smc" : "OASMC",
    "digs" : "DiGS",
    "mala" : "MALA",
    "hmc" : "HMC",
    "gmnu_2_40": "GMNU-2",
    "gm_2_40": "GM-2",
    "gmnu_16_40": "GMNU-16",
    "gm_16_40": "GM-16",
    "lennard_jones_13": "LJ-13",
    "lennard_jones_55": "LJ-55",
    "aldp_vacuum": "ALDP",
    "mlp_posterior": "BNN",
    "test/w2_data": "$W_2$",
    "test/mmd_energy": "MMD",
    "test/tvd_energy": "TVD",
    "test/w2_energy": "$W_2$ density",
    "test/relative_mae": "Relative MAE",
    "test/w2_data_equivariant": "$W_2$",
    "test/rel_mae_equivariant": "Relative MAE",
    "test/kl_div_ramachandran": "Ramachandran KL",
    "test/test_nll": "Test NLL",
    "num_density_evals": "Density evaluations",
}


n_cols = len(samples_dict)+1
fig, axs = plt.subplots(1, n_cols, figsize=(4*n_cols, 4))

plot_results(axs[0], gt_samples)
axs[0].set_title("Ground Truth", fontsize=25)
for i in range(0, len(samples_dict)):
    ax = axs[i+1]
    sampler_name = list(samples_dict.keys())[i]
    samples = samples_dict[sampler_name]    
    plot_results(ax, samples)
    ax.set_title(NAMES_TO_SHOW[sampler_name], fontsize=25)
plt.tight_layout()
plt.savefig(SAVE_PATH + f"samples_gmnu2.pdf", dpi=300, bbox_inches='tight')
plt.show()